# 足球测试第4轮：管理器创建与启停

对应[操作与记录表](足球测试_第4轮_操作与记录.md)。本轮实际创建管理器，再测试两次启停，第二次复用同一对象。TF最多重试三次；每项的异常分别保存。

使用独立内核，结束其他运动程序，取走足球。STEP 04、05会实际开启视觉踢球控制，每次观察三秒后尝试stop。本轮不下发球位、方向或力度。发生stop报错或现场异常，按已有人工流程接管，结束本轮。

## STEP 01　连接并建立记录

运行一次。连接成功后，即使模式读取失败，也会保留连接继续测试。重复运行此格会提示沿用已有连接，保持原目录。

重新开始整轮前，先确认控制已停止、机器人已人工恢复，再通过Jupyter的“内核→重启内核”清理Python连接。内核重启不能代替机器人停止。

In [ ]:
TESTER = "xy"
ROBOT_ID = "060100"
FIRMWARE = "1.8.1"

import sys
import json
import time
import math
import inspect
import traceback
import importlib.metadata
from datetime import datetime
from pathlib import Path
import numpy as np

if "robot" in globals() or "report" in globals():
    raise RuntimeError("已有本轮连接或记录，不重复初始化；请继续后续格")
OUTPUT = Path.cwd() / "足球测试结果" / datetime.now().strftime("第4轮_%Y%m%d_%H%M%S_%f")
OUTPUT.mkdir(parents=True, exist_ok=False)
report = {"test": "足球第4轮_实际创建与启停_v1", "created_at": datetime.now().isoformat(),
          "tester": TESTER, "robot_id": ROBOT_ID, "firmware_note": FIRMWARE,
          "python": sys.version, "capabilities": {}, "cycles": {}, "errors": []}

def json_safe(value):
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, np.generic):
        return json_safe(value.item())
    if isinstance(value, float) and not math.isfinite(value):
        return repr(value)
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    return value

def save():
    (OUTPUT / "原始记录.json").write_text(
        json.dumps(json_safe(report), ensure_ascii=False, indent=2, default=str, allow_nan=False), encoding="utf-8")

def record_error(stage, exc):
    item = {"stage": stage, "error": repr(exc), "traceback": traceback.format_exc()}
    report["errors"].append(item)
    print(stage, "：", repr(exc))
    save()
    return item

def read_modes(label):
    # 两条读取各自记录；失败不会挡住后面的构造函数或停止调用。
    row = {"label": label}
    try:
        row["public_mode"] = str(robot.get_mode())
    except Exception as exc:
        row["public_mode_error"] = repr(exc)
        record_error(label + "/get_mode", exc)
    try:
        value = robot.loco_client.GetMode().mode
        row["raw_mode"] = str(value)
        row["raw_mode_value"] = getattr(value, "value", None)
    except Exception as exc:
        row["raw_mode_error"] = repr(exc)
        record_error(label + "/GetMode", exc)
    report.setdefault("mode_snapshots", []).append(row)
    save()
    print(label, row)
    return row

try:
    report["sdk_version"] = importlib.metadata.version("boosteros")
    from boosteros.robots.booster import BoosterRobot, SoccerKickManager
    robot = BoosterRobot(enable_tf_listener=True)
except Exception as exc:
    report["capabilities"]["机器人连接"] = "失败"
    record_error("连接", exc)
    raise
report["capabilities"]["机器人连接"] = "成功"
save()
try:
    report["robot_info"] = repr(robot.robot_info)
except Exception as exc:
    record_error("读取robot_info", exc)
read_modes("连接后")
print("SDK：", report["sdk_version"])
print("结果目录：", OUTPUT.resolve())

## STEP 02　复查相机到trunk的TF

先等一秒让新连接接收数据，再调用相同公开接口，最多三次。每次请求或报错都保留；失败也继续STEP 03。

这些等待用于排查缓存时间问题，不能保证解决上一轮外推错误。代码不改SDK内部时间，不把trunk直接当作已确认的踢球坐标原点。

In [ ]:
SOURCE_FRAME = "head_color_optical_frame"
TARGET_FRAME = "trunk"
if "tf_attempts" in report:
    raise RuntimeError("本轮TF已记录，不覆盖；请继续STEP 03")
report["tf_attempts"] = []
report["capabilities"]["相机到trunk变换"] = "未取得"
time.sleep(1.0)
try:
    frames = list(robot.list_frames())
    report["frames"] = frames
    if SOURCE_FRAME not in frames or TARGET_FRAME not in frames:
        report["capabilities"]["相机到trunk变换"] = "坐标系本轮未列出"
        print("坐标系列表：", frames)
    else:
        for attempt in range(1, 4):
            row = {"attempt": attempt, "source_frame": SOURCE_FRAME, "target_frame": TARGET_FRAME}
            try:
                transform = robot.get_transform(target_frame=TARGET_FRAME, source_frame=SOURCE_FRAME)
                row["call_returned"] = True
                matrix = np.asarray(transform.to_matrix())
                row["transform"] = {"source_frame": transform.source_frame,
                                    "target_frame": transform.target_frame,
                                    "header": repr(transform.header),
                                    "translation": np.asarray(transform.translation).tolist(),
                                    "rotation": np.asarray(transform.rotation).tolist(),
                                    "matrix": matrix.tolist()}
                row["matrix_structure_ok"] = bool(matrix.shape == (4, 4) and np.isfinite(matrix).all())
                report["capabilities"]["相机到trunk变换"] = "已取得" if row["matrix_structure_ok"] else "返回结构待核实"
                print("第", attempt, "次：取得返回")
                print(matrix)
            except Exception as exc:
                row["error"] = record_error("TF第" + str(attempt) + "次", exc)
            report["tf_attempts"].append(row)
            save()
            if row.get("call_returned"):
                break
            if attempt < 3:
                time.sleep(0.2)
except Exception as exc:
    record_error("列出TF坐标系", exc)
save()
print("TF结果：", report["capabilities"]["相机到trunk变换"])

## STEP 03　实际创建管理器

下面的`SoccerKickManager(robot)`就是构造调用，不在它前面附加模式读取。程序分别保存“已进入构造调用”和“调用是否返回”，完整报错可以定位到SDK中的具体一行。

成功后再读模式。创建失败则跳过STEP 04、05，直接运行STEP 06；本内核不反复创建。

In [ ]:
if report.get("manager_create_attempted") or "soccer" in globals():
    raise RuntimeError("本轮已尝试创建管理器，不重复创建；请继续后续格")
report["manager_create_attempted"] = True
save()
try:
    soccer = SoccerKickManager(robot)
except Exception as exc:
    report["capabilities"]["实际创建管理器"] = "失败"
    record_error("SoccerKickManager构造", exc)
else:
    report["manager_type"] = type(soccer).__module__ + "." + type(soccer).__name__
    report["capabilities"]["实际创建管理器"] = "成功"
    save()
    print("管理器实际创建成功；尚未启动。")
    read_modes("创建后")
save()
print("创建结果：", report["capabilities"]["实际创建管理器"])

## STEP 04　第一次启动与停止

确认其他运动程序已结束、机器人处于可启动状态，输入“启停”。代码实际执行`start()`，返回后观察三秒，再在`finally`中执行`stop()`。创建失败会自动跳过。

原始模式、两个方法的返回和报错分别保存。请观察机器人实际表现；没有球位和指令时，不期待它踢球。stop返回后可能仍处于足球模式，这是本机实现的行为。stop报错时先人工接管，本轮不继续重启。

In [ ]:
def test_start_stop(label):
    if label in report["cycles"]:
        print("本步已有记录，不重复启动：", label)
        return
    row = {"start_attempted": False, "start_returned": False,
           "stop_attempted": False, "stop_returned": False}
    report["cycles"][label] = row
    save()
    try:
        row["start_attempted"] = True
        report["control_uncertain"] = True
        save()
        soccer.start()
        row["start_returned"] = True
        save()
        print(label, "start已返回；观察三秒，随后调用stop。")
        time.sleep(3.0)
    except (Exception, KeyboardInterrupt) as exc:
        row["start_or_observation_error"] = record_error(label + "/start或观察", exc)
    finally:
        # start报错也尝试关闭，避免它已部分完成模式切换或开启控制。
        row["stop_attempted"] = True
        try:
            soccer.stop()
        except (Exception, KeyboardInterrupt) as exc:
            row["stop_error"] = record_error(label + "/stop", exc)
            print("stop未正常返回，请人工接管，本轮不继续重启。")
        else:
            row["stop_returned"] = True
            report["control_uncertain"] = False
            print(label, "stop已返回；请核对现场状态。")
        save()
    # 模式读取放在停止尝试之后，不能阻断finally中的stop。
    read_modes(label + "停止尝试后")

if report["capabilities"].get("实际创建管理器") != "成功":
    print("创建未成功，跳过启停，继续STEP 06。")
elif "第一次启停" in report["cycles"]:
    print("第一次启停已有记录，不重复启动。")
elif input("已确认可启动，输入“启停”继续：").strip() == "启停":
    test_start_stop("第一次启停")
else:
    print("未启动；可以继续STEP 06保存。")

## STEP 05　停止后再次启动

只有第一次start与stop均正常返回，且你确认现场状态正常后，才输入“重启”。此步复用原来的`soccer`，再次实际调用start与stop；没有创建第二个对象。

第一次启停失败会自动跳过。此时保留错误交助手分析，不用自行反复启停或改SDK。

In [ ]:
first = report["cycles"].get("第一次启停", {})
if not first.get("start_returned") or not first.get("stop_returned") or first.get("start_or_observation_error"):
    print("第一次启停未全部返回，跳过重启，继续STEP 06。")
elif report.get("control_uncertain", True):
    print("控制关闭情况待处理，跳过重启，继续STEP 06。")
elif "停止后重启" in report["cycles"]:
    print("重启已有记录，不重复执行。")
elif input("确认第一次启停后现场正常，输入“重启”继续：").strip() == "重启":
    test_start_stop("停止后重启")
else:
    print("未重启；继续STEP 06保存。")

## STEP 06　保存并交回记录

原样运行。本格会补存模式读取、机型识别和TF适配层的本机实现，供助手定位错误；你无需读源码。取不到源码也会保留原因，照常生成汇总。

把整个结果目录复制回知识库，填写[本轮记录](足球测试_第4轮_操作与记录.md)。如果stop报错或现场异常，先人工恢复机器人，再结束内核。正常结束后也按既有操作切回所需模式。

下一轮仍要测试方向、力度、球位更新与实际踢球；本轮启停返回不能代替这些功能的验证。

In [ ]:
source_records = []
if "robot" in globals():
    objects = [("BoosterRobot_get_mode", type(robot), "get_mode"),
               ("BoosterRobot_get_model", type(robot), "_get_model"),
               ("RosAdapter_get_transform", getattr(robot, "ros_adapter", None), "get_transform")]
    for label, owner, attr in objects:
        row = {"name": label}
        try:
            obj = getattr(owner, attr)
            row["doc"] = inspect.getdoc(obj)
            row["file"] = inspect.getfile(obj)
            filename = label + "_本机实现.txt"
            (OUTPUT / filename).write_text(inspect.getsource(obj), encoding="utf-8")
            row["source_file"] = filename
        except Exception as exc:
            row["error"] = repr(exc)
        source_records.append(row)
report["sdk_sources"] = source_records
save()
lines = ["# 足球第4轮：实际创建与启停", "", "时间：" + report["created_at"],
         "SDK：" + report.get("sdk_version", "未取得"), "",
         "| 检查项 | 结果 |", "| --- | --- |"]
for name in ["机器人连接", "相机到trunk变换", "实际创建管理器"]:
    lines.append("| " + name + " | " + report["capabilities"].get(name, "未执行") + " |")
for label in ["第一次启停", "停止后重启"]:
    row = report["cycles"].get(label)
    status = "未执行" if row is None else (
        "start调用/返回=" + str(row["start_attempted"]) + "/" + str(row["start_returned"])
        + "；stop调用/返回=" + str(row["stop_attempted"]) + "/" + str(row["stop_returned"]))
    lines.append("| " + label + " | " + status + " |")
lines += ["", "## 错误", ""]
for item in report["errors"]:
    lines.append("- " + item["stage"] + "：" + item["error"])
lines += ["", "现场表现见人工记录；启停返回不代表试踢成功。方向/力度及球位更新仍未测试。"]
(OUTPUT / "能力汇总.md").write_text("\n".join(lines), encoding="utf-8")
print("结果目录：", OUTPUT.resolve())
print("请填写第4轮记录，交助手回读。")